In [304]:
import pandas as pd
import re

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

In [305]:
seed = 42

root_path = "/home/stefan/ioai-prep/kits/text_emotion_cls"

# Data

In [306]:
def preprocess_text(text):
    """Clean and normalize text."""
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    # Remove email addresses
    text = re.sub(r"\S+@\S+", "", text)
    # Remove special characters and digits
    text = re.sub(r"[^a-z\s]", "", text)
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [307]:
df = pd.read_csv(f"{root_path}/train.csv").drop(["SampleID"], axis=1)
df["text"] = df["text"].apply(preprocess_text)
df.head()

,text,label
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [308]:
df["label"].value_counts()

label
joy         5362
sadness     4666
anger       2159
fear        1937
love        1304
surprise     572
Name: count, dtype: int64

# Model

In [309]:
X_train, X_test, y_train, y_test = train_test_split(df["text"], df["label"], stratify=df["label"], random_state=seed)

In [310]:
tfidf = TfidfVectorizer(
    min_df=2,
    max_df=0.95,
    max_features=10000,
    ngram_range=(1, 2),
    stop_words="english",
    sublinear_tf=True,
    norm="l2",
)

X_train = tfidf.fit_transform(X_train)
X_test = tfidf.transform(X_test)

In [311]:
def evaluate(model):
    cv = cross_val_score(
        model, X_train, y_train, scoring="f1_macro", cv=3, n_jobs=-1
    )
    cv_score = -(cv.mean() - cv.std()).item()

    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    score = f1_score(y_test, preds, average='macro')

    return -cv_score, score

In [312]:
ridge = RidgeClassifier()

evaluate(ridge)

(0.8530558407642598, 0.8719136963158179)

In [318]:
svc = LinearSVC(max_iter=2000, random_state=seed)

evaluate(svc)

(0.8539316422596103, 0.8704634394352372)

In [319]:
model = svc

model.fit(X_train, y_train)

LinearSVC(max_iter=2000, random_state=42)

# Submission

In [320]:
df_test = pd.read_csv(f"{root_path}/test.csv")
submission = df_test[["SampleID"]]

In [321]:
X = tfidf.transform(df_test["text"])
submission["label"] = model.predict(X)

In [322]:
submission.to_csv(f"{root_path}/submission.csv", index=False)
submission.head()

,SampleID,label
0,16001,sadness
1,16002,sadness
2,16003,sadness
3,16004,joy
4,16005,sadness
